# RFUAV Experiment 4 — Kaggle Version

This notebook runs two experiments on the Kaggle **Noisy Drone RF Signal Classification** dataset:

1. **Spectrogram model:** `x_spec → ResNet18 + learning-rate scheduler`
2. **IQ model:** `x_iq → 1D CNN + learning-rate scheduler`

Before running:

1. Open the notebook on Kaggle.
2. Click **Add Input**.
3. Add the dataset: `sgluege/noisy-drone-rf-signal-classification`.
4. In notebook settings, set **Accelerator → GPU**.

The notebook uses:

```text
/kaggle/input/    # read-only dataset files
/kaggle/working/  # saved models, plots, CSV results
```

The dataset provides two useful inputs:

```text
x_spec: [number_of_samples, 2, 128, 128]    # spectrogram tensor
x_iq:   [number_of_samples, 2, 16384]       # raw I/Q signal tensor
```

The goal is to compare whether a spectrogram-based model or a direct IQ-based model performs better, especially at low SNR.


## 1. Check GPU, RAM, and disk

Run this first to confirm whether Kaggle has a GPU attached and enough RAM/disk space.

In [ ]:
import os
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected. On Kaggle: Settings → Accelerator → GPU.")

print("\nRAM:")
!free -h

print("\nDisk:")
!df -h /kaggle/working


## 2. Locate the Kaggle dataset

This replaces the old Colab/Kaggle-API download cells. On Kaggle, use **Add Input** instead of downloading the dataset inside the notebook.

In [ ]:
import os
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")
SAVE_DIR = Path("/kaggle/working/noisy_drone_resnet18_spec_balanced")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# Find dataset.pt automatically so the notebook still works if Kaggle changes the mounted folder name.
candidates = list(KAGGLE_INPUT.rglob("dataset.pt"))
if not candidates:
    raise FileNotFoundError(
        "Could not find dataset.pt under /kaggle/input. "
        "Click Add Input and attach sgluege/noisy-drone-rf-signal-classification first."
    )

DATA_PATH = candidates[0]
DATA_DIR = DATA_PATH.parent

class_stats_path = DATA_DIR / "class_stats.csv"
snr_stats_path = DATA_DIR / "SNR_stats.csv"

print("DATA_DIR:", DATA_DIR)
print("DATA_PATH:", DATA_PATH)
print("File size GB:", DATA_PATH.stat().st_size / 1024**3)
print("SAVE_DIR:", SAVE_DIR)

print("\nFiles in DATA_DIR:")
for p in sorted(DATA_DIR.iterdir()):
    print("-", p.name)


## 3. Inspect class and SNR statistics

This shows the class imbalance and SNR distribution.


In [ ]:
import pandas as pd

class_stats = pd.read_csv(class_stats_path)
snr_stats = pd.read_csv(snr_stats_path)

print("Class stats:")
display(class_stats)

print("SNR stats:")
display(snr_stats)


## 4. Safely load `dataset.pt` with memory mapping

`dataset.pt` is large, so normal `torch.load()` can use too much RAM. This notebook uses `mmap=True` to avoid loading everything into memory at once.

In [ ]:
import torch

print("Before loading:")
!free -h

data = torch.load(
    DATA_PATH,
    map_location="cpu",
    mmap=True,
    weights_only=False
)

print("Loaded successfully with mmap")
print("Type:", type(data))

for k, v in data.items():
    print(k, type(v), getattr(v, "shape", None), getattr(v, "dtype", None))

print("After loading:")
!free -h


## 5. Create a balanced 7-class subset

The full dataset is imbalanced because the `Noise` class is much larger than the smallest class.

For a fair first experiment, use:

- 1700 training samples per class
- 400 validation samples per class
- 7 classes total

In [ ]:
import numpy as np
import random

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

class_names = [
    "DJI",
    "FutabaT14",
    "FutabaT7",
    "Graupner",
    "Noise",
    "Taranis",
    "Turnigy"
]

y = data["y"]

# Set QUICK_TEST = True for a short pipeline test before using GPU quota on a full run.
QUICK_TEST = True

if QUICK_TEST:
    train_per_class = 300
    valid_per_class = 100
else:
    train_per_class = 1700
    valid_per_class = 400

train_indices = []
valid_indices = []

for cls, name in enumerate(class_names):
    cls_indices = torch.where(y == cls)[0].cpu().numpy()
    np.random.shuffle(cls_indices)

    needed = train_per_class + valid_per_class
    if len(cls_indices) < needed:
        raise ValueError(f"Class {name} has only {len(cls_indices)} samples, but {needed} are needed.")

    train_indices.extend(cls_indices[:train_per_class])
    valid_indices.extend(cls_indices[train_per_class:train_per_class + valid_per_class])

train_indices = np.array(train_indices)
valid_indices = np.array(valid_indices)

np.random.shuffle(train_indices)
np.random.shuffle(valid_indices)

print("Train samples:", len(train_indices))
print("Valid samples:", len(valid_indices))

for cls, name in enumerate(class_names):
    train_count = int((y[train_indices] == cls).sum().item())
    valid_count = int((y[valid_indices] == cls).sum().item())
    print(f"{name}: train={train_count}, valid={valid_count}")


## 6. Build a PyTorch Dataset and DataLoader for `x_spec`

`x_spec` has shape:

```text
[number_of_samples, 2, 128, 128]
```

So the CNN input channel count must be `2`.


In [ ]:
from torch.utils.data import Dataset, DataLoader

class NoisyDroneSpecDataset(Dataset):
    def __init__(self, data, indices):
        self.x_spec = data["x_spec"]
        self.y = data["y"]
        self.snr = data["snr"]
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = int(self.indices[idx])
        x = self.x_spec[real_idx]   # [2, 128, 128]
        label = self.y[real_idx]
        snr = self.snr[real_idx]
        return x, label, snr

batch_size = 64
num_workers = 0  # Safer for Kaggle + memory-mapped tensors
pin_memory = torch.cuda.is_available()

train_dataset = NoisyDroneSpecDataset(data, train_indices)
valid_dataset = NoisyDroneSpecDataset(data, valid_indices)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=pin_memory
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory
)

x_batch, y_batch, snr_batch = next(iter(train_loader))
print("x batch:", x_batch.shape)
print("y batch:", y_batch.shape)
print("snr batch:", snr_batch.shape)
print("x dtype:", x_batch.dtype)
print("y dtype:", y_batch.dtype)
print("snr dtype:", snr_batch.dtype)


## 7. Define ResNet18 for 2-channel RF spectrograms

Normal ResNet18 expects RGB images with shape:

```text
[batch, 3, height, width]
```

This dataset uses RF spectrogram tensors with shape:

```text
[batch, 2, 128, 128]
```

So we modify the first convolution layer to accept 2 input channels and change the final fully connected layer to output 7 classes.

For this RF dataset, we use `weights=None` because ImageNet pretrained weights are designed for natural RGB images, not 2-channel RF spectrograms.


In [ ]:
import torch.nn as nn
from torchvision import models

class ResNet18RFSpec(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()

        # Start with a standard ResNet18 architecture.
        # weights=None avoids RGB ImageNet pretrained weights.
        self.model = models.resnet18(weights=None)

        # Original ResNet18 first layer expects 3-channel RGB images.
        # Our RF spectrogram tensors have 2 channels, so change 3 -> 2.
        self.model.conv1 = nn.Conv2d(
            in_channels=2,
            out_channels=64,
            kernel_size=7,
            stride=2,
            padding=3,
            bias=False
        )

        # Replace the final classifier for 7 drone/noise classes.
        self.model.fc = nn.Linear(self.model.fc.in_features, num_classes)

    def forward(self, x):
        return self.model(x)

# Quick shape test before training.
model_test = ResNet18RFSpec(num_classes=7)
with torch.no_grad():
    out = model_test(x_batch[:2])
print("Test output shape:", out.shape)


## 8. Set Kaggle output folder for saving results

Kaggle outputs should be saved in `/kaggle/working/` so they appear in the notebook Output section.


In [ ]:
SAVE_DIR = Path("/kaggle/working/noisy_drone_resnet18_spec_balanced")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print("Save directory:", SAVE_DIR)


## 9. Train ResNet18 with learning-rate scheduling

This trains ResNet18 on the balanced spectrogram subset and saves:

- `best.pt`
- `last.pt`
- `history.csv`

This version uses:

- `AdamW` optimizer
- `CosineAnnealingLR` scheduler

Start with `batch_size = 64`. If Kaggle gives a CUDA out-of-memory error, go back to the DataLoader cell and reduce `batch_size` to `32`.


In [ ]:
import torch.optim as optim
from tqdm import tqdm
import pandas as pd

SPEC_SAVE_DIR = Path("/kaggle/working/noisy_drone_resnet18_spec_balanced_scheduler")
SPEC_SAVE_DIR.mkdir(parents=True, exist_ok=True)
SAVE_DIR = SPEC_SAVE_DIR  # keep this name for the existing evaluation cells below

# If True, train only a tiny version to check that the notebook runs.
# Set QUICK_TEST = False in the balanced subset cell for a real 20-epoch run.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("Saving spectrogram model to:", SPEC_SAVE_DIR)

model = ResNet18RFSpec(num_classes=len(class_names)).to(device)
criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

epochs = 2 if QUICK_TEST else 20
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=epochs
)

history = []
best_acc = 0.0

for epoch in range(epochs):
    print("=" * 60)
    print(f"Epoch {epoch + 1}/{epochs}")

    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for x, labels, snrs in tqdm(train_loader, desc="Training"):
        x = x.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * x.size(0)
        preds = outputs.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total

    model.eval()
    valid_loss = 0.0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():
        for x, labels, snrs in tqdm(valid_loader, desc="Validation"):
            x = x.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(x)
            loss = criterion(outputs, labels)

            valid_loss += loss.item() * x.size(0)
            preds = outputs.argmax(dim=1)
            valid_correct += (preds == labels).sum().item()
            valid_total += labels.size(0)

    valid_loss /= valid_total
    valid_acc = valid_correct / valid_total

    scheduler.step()
    current_lr = optimizer.param_groups[0]["lr"]

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Acc : {train_acc:.4f}")
    print(f"Valid Loss: {valid_loss:.4f}")
    print(f"Valid Acc : {valid_acc:.4f}")
    print(f"Learning Rate: {current_lr:.6f}")

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "valid_loss": valid_loss,
        "valid_acc": valid_acc,
        "lr": current_lr,
    })

    if valid_acc > best_acc:
        best_acc = valid_acc
        torch.save(model.state_dict(), SPEC_SAVE_DIR / "best.pt")
        print("Saved best model.")

torch.save(model.state_dict(), SPEC_SAVE_DIR / "last.pt")

history_df = pd.DataFrame(history)
history_df.to_csv(SPEC_SAVE_DIR / "history.csv", index=False)

print("Finished spectrogram ResNet18 training.")
print("Best valid accuracy:", best_acc)
print("Saved to:", SPEC_SAVE_DIR)


## 10. Plot training curves


In [ ]:
import matplotlib.pyplot as plt

history_path = os.path.join(SAVE_DIR, "history.csv")
history_df = pd.read_csv(history_path)
display(history_df)

plt.figure(figsize=(8, 5))
plt.plot(history_df["epoch"], history_df["train_acc"], label="Train Accuracy")
plt.plot(history_df["epoch"], history_df["valid_acc"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Noisy Drone RF - ResNet18 Spectrogram Accuracy")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history_df["epoch"], history_df["train_loss"], label="Train Loss")
plt.plot(history_df["epoch"], history_df["valid_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Noisy Drone RF - ResNet18 Spectrogram Loss")
plt.legend()
plt.grid(True)
plt.show()

if "lr" in history_df.columns:
    plt.figure(figsize=(8, 5))
    plt.plot(history_df["epoch"], history_df["lr"], marker="o")
    plt.xlabel("Epoch")
    plt.ylabel("Learning Rate")
    plt.title("ResNet18 Spectrogram Learning-Rate Schedule")
    plt.grid(True)
    plt.show()


## 11. Classification report and confusion matrix

Use macro F1 because this dataset is naturally imbalanced, even though this first subset is balanced.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

# Load best model before evaluation
best_path = os.path.join(SAVE_DIR, "best.pt")
if os.path.exists(best_path):
    model.load_state_dict(torch.load(best_path, map_location=device))
    print("Loaded best model:", best_path)

model.eval()
all_preds = []
all_labels = []
all_snrs = []

with torch.no_grad():
    for x, labels, snrs in valid_loader:
        x = x.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        outputs = model(x)
        preds = outputs.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
        all_snrs.extend(snrs.cpu().numpy())

print(classification_report(all_labels, all_preds, target_names=class_names))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 8))
plt.imshow(cm)
plt.title("Noisy Drone RF - ResNet18 Spectrogram Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(np.arange(len(class_names)), class_names, rotation=45, ha="right")
plt.yticks(np.arange(len(class_names)), class_names)
plt.colorbar()

for i in range(len(class_names)):
    for j in range(len(class_names)):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.tight_layout()
cm_path = os.path.join(SAVE_DIR, "confusion_matrix.png")
plt.savefig(cm_path, dpi=200)
plt.show()
print("Saved confusion matrix to:", cm_path)


## 12. Accuracy by SNR

This is one of the most important analyses for this dataset.

Low SNR values like `-20 dB` should be harder. High SNR values like `20 dB` to `30 dB` should be easier.


In [ ]:
results_df = pd.DataFrame({
    "label": all_labels,
    "pred": all_preds,
    "snr": all_snrs,
})
results_df["correct"] = results_df["label"] == results_df["pred"]

snr_acc = results_df.groupby("snr")["correct"].mean().reset_index()
snr_acc.columns = ["SNR", "accuracy"]
display(snr_acc)

plt.figure(figsize=(9, 5))
plt.plot(snr_acc["SNR"], snr_acc["accuracy"], marker="o")
plt.xlabel("SNR (dB)")
plt.ylabel("Accuracy")
plt.title("Validation Accuracy by SNR")
plt.grid(True)
plt.tight_layout()
snr_path = os.path.join(SAVE_DIR, "accuracy_by_snr.png")
plt.savefig(snr_path, dpi=200)
plt.show()
print("Saved SNR accuracy plot to:", snr_path)


## 13. Confusion matrices by SNR group

The overall confusion matrix hides where the model fails. This section splits the validation set into low, medium, and high SNR groups, then creates a separate confusion matrix for each group.

Suggested groups:

```text
Low SNR:    -20 to -10 dB
Medium SNR: -8 to 4 dB
High SNR:    6 to 30 dB
```

This helps show whether errors such as `Turnigy → Noise` mainly happen when the RF signal is weak.


In [ ]:

from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

# This cell expects these variables from the evaluation cell:
# all_labels, all_preds, all_snrs, class_names, SAVE_DIR

results_df = pd.DataFrame({
    "label": np.array(all_labels),
    "pred": np.array(all_preds),
    "snr": np.array(all_snrs),
})

results_df["correct"] = results_df["label"] == results_df["pred"]

snr_groups = {
    "low_snr_-20_to_-10": (-20, -10),
    "medium_snr_-8_to_4": (-8, 4),
    "high_snr_6_to_30": (6, 30),
}

def plot_confusion_matrix_for_subset(df_subset, title, save_path):
    labels = list(range(len(class_names)))
    cm = confusion_matrix(df_subset["label"], df_subset["pred"], labels=labels)

    plt.figure(figsize=(8, 8))
    plt.imshow(cm)
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.xticks(np.arange(len(class_names)), class_names, rotation=45, ha="right")
    plt.yticks(np.arange(len(class_names)), class_names)
    plt.colorbar()

    for i in range(len(class_names)):
        for j in range(len(class_names)):
            plt.text(j, i, int(cm[i, j]), ha="center", va="center")

    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.show()

    return cm

snr_summary_rows = []

for group_name, (snr_min, snr_max) in snr_groups.items():
    subset = results_df[(results_df["snr"] >= snr_min) & (results_df["snr"] <= snr_max)].copy()

    if len(subset) == 0:
        print(f"No validation samples found for {group_name}")
        continue

    acc = subset["correct"].mean()
    print("=" * 80)
    print(f"{group_name}: SNR {snr_min} to {snr_max} dB")
    print(f"Samples: {len(subset)}")
    print(f"Accuracy: {acc:.4f}")

    print("\nClassification report:")
    print(classification_report(
        subset["label"],
        subset["pred"],
        labels=list(range(len(class_names))),
        target_names=class_names,
        zero_division=0,
    ))

    save_path = os.path.join(SAVE_DIR, f"confusion_matrix_{group_name}.png")
    cm_group = plot_confusion_matrix_for_subset(
        subset,
        f"Noisy Drone RF - Confusion Matrix ({group_name})",
        save_path
    )
    print("Saved:", save_path)

    snr_summary_rows.append({
        "group": group_name,
        "snr_min": snr_min,
        "snr_max": snr_max,
        "samples": len(subset),
        "accuracy": acc,
    })

snr_group_summary = pd.DataFrame(snr_summary_rows)
display(snr_group_summary)

summary_path = os.path.join(SAVE_DIR, "snr_group_summary.csv")
snr_group_summary.to_csv(summary_path, index=False)
print("Saved SNR group summary to:", summary_path)


---

# 14. IQ-based model experiment

The previous experiment used `x_spec`, a 2-channel spectrogram tensor.

This section uses the raw I/Q tensor directly:

```text
x_iq: [number_of_samples, 2, 16384]
```

The model is a 1D CNN with `Conv1d(2, ...)`, because the two channels represent I and Q. This experiment also uses `AdamW` and `CosineAnnealingLR`.


## 14.1 Build Dataset and DataLoaders for `x_iq`

Each sample is normalized independently. This helps the 1D CNN focus on waveform shape instead of absolute amplitude scale.


In [ ]:
class NoisyDroneIQDataset(Dataset):
    def __init__(self, data, indices, normalize=True):
        self.x_iq = data["x_iq"]
        self.y = data["y"]
        self.snr = data["snr"]
        self.indices = indices
        self.normalize = normalize

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = int(self.indices[idx])

        x = self.x_iq[real_idx]   # [2, 16384]
        label = self.y[real_idx]
        snr = self.snr[real_idx]

        if self.normalize:
            x = (x - x.mean()) / (x.std() + 1e-8)

        return x, label, snr

# Raw IQ sequences are larger than spectrogram tensors, so reduce this if GPU memory is tight.
iq_batch_size = 64

train_iq_dataset = NoisyDroneIQDataset(data, train_indices, normalize=True)
valid_iq_dataset = NoisyDroneIQDataset(data, valid_indices, normalize=True)

train_iq_loader = DataLoader(
    train_iq_dataset,
    batch_size=iq_batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=pin_memory
)

valid_iq_loader = DataLoader(
    valid_iq_dataset,
    batch_size=iq_batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=pin_memory
)

x_iq_batch, y_iq_batch, snr_iq_batch = next(iter(train_iq_loader))

print("IQ batch:", x_iq_batch.shape)
print("Label batch:", y_iq_batch.shape)
print("SNR batch:", snr_iq_batch.shape)
print("IQ dtype:", x_iq_batch.dtype)


## 14.2 Define IQ 1D CNN

This model learns directly from the I/Q time sequence instead of the spectrogram image.


In [ ]:
class IQ1DCNN(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(2, 32, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(32, 64, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(64, 128, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.MaxPool1d(2),

            nn.Conv1d(128, 256, kernel_size=5, stride=2, padding=2),
            nn.BatchNorm1d(256),
            nn.ReLU(),

            nn.AdaptiveAvgPool1d(1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Quick shape test before training.
model_test = IQ1DCNN(num_classes=len(class_names))
with torch.no_grad():
    out = model_test(x_iq_batch[:2])

print("Test output shape:", out.shape)


## 14.3 Train IQ 1D CNN with learning-rate scheduling

This uses the same balanced indices as the spectrogram experiment, so the two results are comparable.


In [ ]:
IQ_SAVE_DIR = Path("/kaggle/working/noisy_drone_iq_1dcnn_balanced_scheduler")
IQ_SAVE_DIR.mkdir(parents=True, exist_ok=True)

print("Using device:", device)
print("Saving IQ model to:", IQ_SAVE_DIR)

model_iq = IQ1DCNN(num_classes=len(class_names)).to(device)
criterion_iq = nn.CrossEntropyLoss()

optimizer_iq = optim.AdamW(
    model_iq.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

iq_epochs = 2 if QUICK_TEST else 20
scheduler_iq = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_iq,
    T_max=iq_epochs
)

history_iq = []
best_iq_acc = 0.0

for epoch in range(iq_epochs):
    print("=" * 60)
    print(f"IQ Epoch {epoch + 1}/{iq_epochs}")

    model_iq.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for x, labels, snrs in tqdm(train_iq_loader, desc="IQ Training"):
        x = x.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer_iq.zero_grad()
        outputs = model_iq(x)
        loss = criterion_iq(outputs, labels)
        loss.backward()
        optimizer_iq.step()

        train_loss += loss.item() * x.size(0)
        preds = outputs.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total

    model_iq.eval()
    valid_loss = 0.0
    valid_correct = 0
    valid_total = 0

    with torch.no_grad():
        for x, labels, snrs in tqdm(valid_iq_loader, desc="IQ Validation"):
            x = x.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model_iq(x)
            loss = criterion_iq(outputs, labels)

            valid_loss += loss.item() * x.size(0)
            preds = outputs.argmax(dim=1)
            valid_correct += (preds == labels).sum().item()
            valid_total += labels.size(0)

    valid_loss /= valid_total
    valid_acc = valid_correct / valid_total

    scheduler_iq.step()
    current_lr = optimizer_iq.param_groups[0]["lr"]

    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Acc : {train_acc:.4f}")
    print(f"Valid Loss: {valid_loss:.4f}")
    print(f"Valid Acc : {valid_acc:.4f}")
    print(f"Learning Rate: {current_lr:.6f}")

    history_iq.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "valid_loss": valid_loss,
        "valid_acc": valid_acc,
        "lr": current_lr,
    })

    if valid_acc > best_iq_acc:
        best_iq_acc = valid_acc
        torch.save(model_iq.state_dict(), IQ_SAVE_DIR / "best.pt")
        print("Saved best IQ model.")

torch.save(model_iq.state_dict(), IQ_SAVE_DIR / "last.pt")

history_iq_df = pd.DataFrame(history_iq)
history_iq_df.to_csv(IQ_SAVE_DIR / "history.csv", index=False)

print("Finished IQ 1D CNN training.")
print("Best IQ valid accuracy:", best_iq_acc)
print("Saved to:", IQ_SAVE_DIR)


## 14.4 Plot IQ training curves


In [ ]:
history_iq_path = IQ_SAVE_DIR / "history.csv"
history_iq_df = pd.read_csv(history_iq_path)
display(history_iq_df)

plt.figure(figsize=(8, 5))
plt.plot(history_iq_df["epoch"], history_iq_df["train_acc"], label="Train Accuracy")
plt.plot(history_iq_df["epoch"], history_iq_df["valid_acc"], label="Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.title("Noisy Drone RF - IQ 1D CNN Accuracy")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history_iq_df["epoch"], history_iq_df["train_loss"], label="Train Loss")
plt.plot(history_iq_df["epoch"], history_iq_df["valid_loss"], label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Noisy Drone RF - IQ 1D CNN Loss")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(history_iq_df["epoch"], history_iq_df["lr"], marker="o")
plt.xlabel("Epoch")
plt.ylabel("Learning Rate")
plt.title("IQ 1D CNN Learning-Rate Schedule")
plt.grid(True)
plt.show()


## 14.5 Evaluate IQ 1D CNN

This creates the same outputs as the spectrogram experiment: classification report, confusion matrix, and saved prediction table.


In [ ]:
# Load best IQ model before evaluation.
best_iq_path = IQ_SAVE_DIR / "best.pt"
if best_iq_path.exists():
    model_iq.load_state_dict(torch.load(best_iq_path, map_location=device))
    print("Loaded best IQ model:", best_iq_path)

model_iq.eval()
all_iq_preds = []
all_iq_labels = []
all_iq_snrs = []

with torch.no_grad():
    for x, labels, snrs in valid_iq_loader:
        x = x.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        outputs = model_iq(x)
        preds = outputs.argmax(dim=1)

        all_iq_preds.extend(preds.cpu().numpy())
        all_iq_labels.extend(labels.cpu().numpy())
        all_iq_snrs.extend(snrs.cpu().numpy())

print(classification_report(
    all_iq_labels,
    all_iq_preds,
    target_names=class_names
))

cm_iq = confusion_matrix(all_iq_labels, all_iq_preds)

plt.figure(figsize=(8, 8))
plt.imshow(cm_iq)
plt.title("Noisy Drone RF - IQ 1D CNN Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.xticks(np.arange(len(class_names)), class_names, rotation=45, ha="right")
plt.yticks(np.arange(len(class_names)), class_names)
plt.colorbar()

for i in range(len(class_names)):
    for j in range(len(class_names)):
        plt.text(j, i, cm_iq[i, j], ha="center", va="center")

plt.tight_layout()
cm_iq_path = IQ_SAVE_DIR / "confusion_matrix_iq_1dcnn.png"
plt.savefig(cm_iq_path, dpi=200)
plt.show()

print("Saved IQ confusion matrix to:", cm_iq_path)

results_iq_df = pd.DataFrame({
    "label": all_iq_labels,
    "pred": all_iq_preds,
    "snr": all_iq_snrs,
})
results_iq_df["correct"] = results_iq_df["label"] == results_iq_df["pred"]
results_iq_df.to_csv(IQ_SAVE_DIR / "valid_predictions.csv", index=False)
print("Saved IQ prediction table to:", IQ_SAVE_DIR / "valid_predictions.csv")


## 14.6 IQ accuracy by SNR


In [ ]:
snr_iq_acc = results_iq_df.groupby("snr")["correct"].mean().reset_index()
snr_iq_acc.columns = ["SNR", "accuracy"]
display(snr_iq_acc)

plt.figure(figsize=(9, 5))
plt.plot(snr_iq_acc["SNR"], snr_iq_acc["accuracy"], marker="o")
plt.xlabel("SNR (dB)")
plt.ylabel("Accuracy")
plt.title("IQ 1D CNN - Validation Accuracy by SNR")
plt.grid(True)
plt.tight_layout()
snr_iq_path = IQ_SAVE_DIR / "accuracy_by_snr_iq_1dcnn.png"
plt.savefig(snr_iq_path, dpi=200)
plt.show()

print("Saved IQ SNR accuracy plot to:", snr_iq_path)


## 14.7 IQ confusion matrices by SNR group

This lets you compare whether the IQ model handles low-SNR samples better or worse than the spectrogram ResNet18 model.


In [ ]:
def plot_iq_confusion_matrix_for_subset(df_subset, title, save_path):
    labels = list(range(len(class_names)))
    cm = confusion_matrix(df_subset["label"], df_subset["pred"], labels=labels)

    plt.figure(figsize=(8, 8))
    plt.imshow(cm)
    plt.title(title)
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.xticks(np.arange(len(class_names)), class_names, rotation=45, ha="right")
    plt.yticks(np.arange(len(class_names)), class_names)
    plt.colorbar()

    for i in range(len(class_names)):
        for j in range(len(class_names)):
            plt.text(j, i, cm[i, j], ha="center", va="center")

    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.show()

    print("Saved:", save_path)
    return cm

for group_name, (low, high) in snr_groups.items():
    subset = results_iq_df[(results_iq_df["snr"] >= low) & (results_iq_df["snr"] <= high)]

    print("=" * 60)
    print(group_name)
    print("Samples:", len(subset))

    if len(subset) == 0:
        print("No samples in this SNR group.")
        continue

    print("Accuracy:", subset["correct"].mean())
    print(classification_report(
        subset["label"],
        subset["pred"],
        labels=list(range(len(class_names))),
        target_names=class_names,
        zero_division=0
    ))

    save_path = IQ_SAVE_DIR / f"confusion_matrix_iq_{group_name}.png"
    plot_iq_confusion_matrix_for_subset(
        subset,
        f"Noisy Drone RF - IQ Confusion Matrix ({group_name})",
        save_path
    )


---

# 15. Compare spectrogram ResNet18 vs IQ 1D CNN

Use this section after both models have finished training and evaluation. The most important comparison is not only overall validation accuracy, but also low-SNR behavior.


In [ ]:
comparison_rows = []

# Spectrogram model metrics from results_df
if "results_df" in globals():
    comparison_rows.append({
        "model": "ResNet18 on x_spec",
        "overall_accuracy": results_df["correct"].mean(),
        "low_snr_accuracy": results_df[(results_df["snr"] >= -20) & (results_df["snr"] <= -10)]["correct"].mean(),
        "medium_snr_accuracy": results_df[(results_df["snr"] >= -8) & (results_df["snr"] <= 4)]["correct"].mean(),
        "high_snr_accuracy": results_df[(results_df["snr"] >= 6) & (results_df["snr"] <= 30)]["correct"].mean(),
    })

# IQ model metrics from results_iq_df
if "results_iq_df" in globals():
    comparison_rows.append({
        "model": "1D CNN on x_iq",
        "overall_accuracy": results_iq_df["correct"].mean(),
        "low_snr_accuracy": results_iq_df[(results_iq_df["snr"] >= -20) & (results_iq_df["snr"] <= -10)]["correct"].mean(),
        "medium_snr_accuracy": results_iq_df[(results_iq_df["snr"] >= -8) & (results_iq_df["snr"] <= 4)]["correct"].mean(),
        "high_snr_accuracy": results_iq_df[(results_iq_df["snr"] >= 6) & (results_iq_df["snr"] <= 30)]["correct"].mean(),
    })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)
comparison_df.to_csv(Path("/kaggle/working/model_comparison_spec_vs_iq.csv"), index=False)
print("Saved comparison table to /kaggle/working/model_comparison_spec_vs_iq.csv")
